# Lesson 4 - LIDAR

The RPLIDAR A1 spins and measures distance in every direction. It appears as a
serial device at `/dev/ttyUSB0`.

We start with the four-sector summary, then draw a full 360 degree radar plot.

In [ ]:
import sys
sys.path.insert(0, "..")
from jbot import lidar_sectors

lidar_sectors()

## 360 degree radar plot

We grab two full revolutions and plot every measurement on a polar chart. North
(up) is the front of the robot; angles increase clockwise. Color encodes distance.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from rplidar import RPLidar

lidar = RPLidar("/dev/ttyUSB0", baudrate=115200)
angles, dists = [], []
try:
    for i, scan in enumerate(lidar.iter_scans(max_buf_meas=2000)):
        for (_, angle, dist) in scan:
            if dist > 0:
                angles.append(np.deg2rad(angle))
                dists.append(dist / 1000.0)
        if i >= 1:
            break
finally:
    lidar.stop()
    lidar.stop_motor()
    lidar.disconnect()

ax = plt.subplot(111, projection="polar")
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)
sc = ax.scatter(angles, dists, s=6, c=dists, cmap="viridis")
ax.set_title("LIDAR 360 scan (meters)")
plt.colorbar(sc, pad=0.1, label="m")
plt.show()

## Sector bar chart

The same data reduced to the nearest obstacle in each of the four sectors. This is
exactly what the obstacle-avoidance logic uses in the next lesson.

In [ ]:
sectors = lidar_sectors()
names = list(sectors.keys())
values = [sectors[n] if sectors[n] is not None else 0 for n in names]
plt.bar(names, values, color="tab:orange")
plt.ylabel("nearest obstacle (m)")
plt.title("Sector distances")
plt.show()

## Live radar

This keeps the LIDAR open and redraws the polar plot continuously, so you can watch
obstacles move in real time. The plot is rendered off-screen and pushed into an
image widget, which is smooth and gives a working **Stop** button. Press **Stop** to
end the loop and release the LIDAR.

In [ ]:
import threading
import io
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from rplidar import RPLidar

view = widgets.Image(format="png")
stop = widgets.Button(description="Stop", button_style="danger")
display(view, stop)

state = {"run": True}
fig = Figure(figsize=(5, 5))
FigureCanvasAgg(fig)
ax = fig.add_subplot(111, projection="polar")

def render(angles, dists):
    ax.clear()
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_ylim(0, 4)
    ax.scatter(angles, dists, s=6, c=dists, cmap="viridis", vmin=0, vmax=4)
    ax.set_title("LIDAR live (meters)")
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    view.value = buf.getvalue()

def worker():
    lidar = RPLidar("/dev/ttyUSB0", baudrate=115200)
    try:
        for scan in lidar.iter_scans(max_buf_meas=2000):
            if not state["run"]:
                break
            angles = [np.deg2rad(a) for (_, a, d) in scan if d > 0]
            dists = [d / 1000.0 for (_, a, d) in scan if d > 0]
            render(angles, dists)
    finally:
        lidar.stop()
        lidar.stop_motor()
        lidar.disconnect()

stop.on_click(lambda _: state.update(run=False))
threading.Thread(target=worker, daemon=True).start()